In [2]:
import pandas as pd
import re

In [3]:
df = pd.read_csv(r"/content/dataset_train.csv")

In [4]:
df = df[["id_str","full_text", "image_url", "tweet_url", "created_at", "username"]]

In [5]:
#remove tweets from user "Britonomist" -> Kebanyakan politik rant tapi usernya itu engineer maka bisa masuk
df = df[df['username'] != 'Britonomist']

In [6]:
#get random 10
sample = df.sample(n=10)
sample

,id_str,full_text,image_url,tweet_url,created_at,username
38,2017194550738166139,210+ AI Tools to replace your tedious work:\n\...,https://pbs.twimg.com/media/G_6DwNdaoAAGs69.jpg,https://x.com/RAVIKUMARSAHU78/status/201719455...,2026-01-30 11:13:53+00:00,RAVIKUMARSAHU78
17231,1920921278157934877,vibe coding is finding where to prompt as much...,NaN,https://x.com/santisiri/status/192092127815793...,2025-05-09 19:18:16+00:00,santisiri
17296,2002183221862072429,Gemini 3 Flash just dropped — and it pushed me...,https://pbs.twimg.com/amplify_video_thumb/2001...,https://x.com/DrBoyuanQian/status/200218322186...,2025-12-20 01:04:13+00:00,DrBoyuanQian
38144,1934386199793107298,whats the best browser AI chat to vibe code on...,NaN,https://x.com/kartojal/status/1934386199793107298,2025-06-15 23:03:03+00:00,kartojal
10443,1921822726408245391,Backpropagation! #BigData #Analytics #DataScie...,https://pbs.twimg.com/media/GquvlY4WwAAm8u4.jpg,https://x.com/gp_pulipaka/status/1921822726408...,2025-05-12 07:00:18+00:00,gp_pulipaka
24286,1907857391313694947,Now let me vibe code with it 🔥🔥,NaN,https://x.com/MatthewBerman/status/19078573913...,2025-04-03 18:07:03+00:00,MatthewBerman
37961,1944064263863382216,What are your top security checks when vibe co...,NaN,https://x.com/chimereogudu/status/194406426386...,2025-07-12 16:00:14+00:00,chimereogudu
19624,1931489353789829587,@TheRealCamelio @CadeONeill @caffeineai ChatGP...,NaN,https://x.com/timurrami/status/193148935378982...,2025-06-07 23:12:01+00:00,timurrami
31996,1987525887403655470,Explore vibe coding as non-tech teams reshape ...,NaN,https://x.com/imjeromekjerome/status/198752588...,2025-11-09 14:21:12+00:00,imjeromekjerome
32523,1893346140189250017,getting rate limited while vibe coding sucks,NaN,https://x.com/we4v3r/status/1893346140189250017,2025-02-22 17:04:31+00:00,we4v3r


In [7]:
import html
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
import unicodedata


# Pastikan sudah download resource nltk
import nltk
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')

stop_words = set(stopwords.words('english')) # Atau 'indonesian' jika data campur

def preprocess_tweet_v2(text):
    # 1. Decode HTML Entities (Penting untuk X/Twitter data)
    # Ini akan merubah &lt; menjadi < sehingga re.sub simbol bisa bekerja
    text = html.unescape(text)

    text = unicodedata.normalize("NFKD", text)

    text = text.lower()

    # 2. Pola Regex kamu sudah bagus, pertahankan untuk URL, Mentions, Hashtags
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
    text = re.sub(r'@\w+', '', text)
    text = re.sub(r'#\w+', '', text)

    # 3. Handle spesifik Vibe Coding sebelum hapus simbol
    text = re.sub(r'vibe\s+coding', 'vibecoding', text)
    text = re.sub(r'vibe\s+code', 'vibecode', text)
    text = re.sub(r'vibe\s+coded', 'vibecoded', text)
    text = re.sub(r'building', 'build', text)
    text = re.sub(r'built', 'build', text)
    text = re.sub(r'apps', 'app', text)

    #change coding, coded to code
    text = re.sub(r'coding', 'code', text)
    text = re.sub(r'coded', 'code', text)

    #change llms to llm
    text = re.sub(r'llms', 'llm', text)

    #change large language model to llm
    text = re.sub(r'large language model', 'llm', text)

    # 4. Hapus simbol dan angka
    text = re.sub(r'[^\w\s]', ' ', text) # Ganti simbol dengan spasi agar kata tidak menempel
    text = re.sub(r'\d+', '', text)

    # 5. Tokenisasi & Stopwords (The missing piece)
    tokens = word_tokenize(text)
    cleaned_tokens = [w for w in tokens if w not in stop_words and len(w) > 2]

    #6. Pastikan cleaned_tokens tidak mengandung duplicate kata
    cleaned_tokens = list(dict.fromkeys(cleaned_tokens))

    return " ".join(cleaned_tokens)

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


In [8]:
df_preprocessed = df['full_text'].fillna('').apply(preprocess_tweet_v2)
df_preprocessed = pd.concat([df_preprocessed, df['image_url']], axis=1)
df_preprocessed = pd.concat([df_preprocessed, df['created_at']], axis=1)
df_preprocessed = pd.concat([df_preprocessed, df['tweet_url']], axis=1)
df_preprocessed = pd.concat([df_preprocessed, df['id_str']], axis=1)

In [9]:
df_preprocessed = df_preprocessed[df_preprocessed['full_text'].apply(lambda x: len(x.split()) >= 4)]
df_preprocessed = df_preprocessed.drop_duplicates(subset=['full_text'])
df_preprocessed = df_preprocessed[~df_preprocessed['full_text'].str.contains('ukraine')]
df_preprocessed = df_preprocessed[~df_preprocessed['full_text'].str.contains('zimbabwe')]
df_preprocessed = df_preprocessed[~df_preprocessed['full_text'].str.contains('liberals')]
df_preprocessed = df_preprocessed[~df_preprocessed['full_text'].str.contains('free ebook')]
df_preprocessed = df_preprocessed[~df_preprocessed['full_text'].str.contains('coursera')]
df_preprocessed = df_preprocessed[~df_preprocessed['full_text'].str.contains('free book')]
df_preprocessed['created_at'] = pd.to_datetime(df_preprocessed['created_at']).dt.strftime('%B %Y')
df_preprocessed.count()

,0
full_text,35059
image_url,16042
created_at,35059
tweet_url,35059
id_str,35059


In [10]:
#get sample of df preprocessed of 10 to print out
sample = df_preprocessed.sample(n=10)
sample

,full_text,image_url,created_at,tweet_url,id_str
23567,vibe users like vibecode lovable cursor bolt r...,https://pbs.twimg.com/amplify_video_thumb/2023...,February 2026,https://x.com/King0fWealth/status/202307067244...,2023070672441008363
17557,fun story shoulder pain decided build habit tr...,https://pbs.twimg.com/media/GtRYl1bXQAAuaO4.jpg,June 2025,https://x.com/thisiskp_/status/193326893408790...,1933268934087905368
8634,hey everyone know piano videos today want shar...,https://pbs.twimg.com/media/GjCj-78W0AAMhch.jpg,February 2025,https://x.com/3merillon/status/188718972985639...,1887189729856397562
4601,fyi google open jobs use code list things buil...,https://pbs.twimg.com/media/Gyiq9pWXsAEczT-.png,August 2025,https://x.com/patloeber/status/195700202030645...,1957002020306452490
12028,demo trading educational tool years fintech wo...,https://pbs.twimg.com/amplify_video_thumb/2027...,February 2026,https://x.com/traderprad/status/20274357514500...,2027435751450022114
36473,man universe want vibecode zooming tech tree,NaN,August 2025,https://x.com/etiennefd/status/195757320343114...,1957573203431145500
28589,since releasing android app forced vibecode vo...,https://pbs.twimg.com/amplify_video_thumb/2016...,January 2026,https://x.com/MaruPelkar/status/20165974007613...,2016597400761356537
37127,power spawn new words lexicon huh within years...,NaN,February 2025,https://x.com/0xf3mi/status/1888291118485303640,1888291118485303640
19446,introduces vibecode daniel roe watch code nuxt...,https://pbs.twimg.com/tweet_video_thumb/Grfjt6...,May 2025,https://x.com/frontendnation/status/1925257746...,1925257746179297777
26189,practical roadmap adopting vibecode emilio sal...,NaN,May 2025,https://x.com/thenewstack/status/1923419224124...,1923419224124309764


In [11]:
df_preprocessed.to_csv(r"./preprocessed_tweets.csv", index=False)